# Buffered Loading with Shuffle Buffer — Solution

In [ ]:
import torch
import random
from torch.utils.data import IterableDataset, DataLoader

class SequentialDataset(IterableDataset):
    def __init__(self, N):
        self.N = N
    def __iter__(self):
        for i in range(self.N):
            yield torch.tensor(i, dtype=torch.long), torch.tensor(float(i))

In [ ]:
class ShuffleBufferDataset(IterableDataset):
    """Wraps an IterableDataset and shuffles its output via a fixed-size buffer."""

    def __init__(self, base_dataset: IterableDataset, buffer_size: int):
        super().__init__()
        if buffer_size < 1:
            raise ValueError("buffer_size must be at least 1")
        self.base_dataset = base_dataset
        self.buffer_size = buffer_size

    def __iter__(self):
        it = iter(self.base_dataset)

        # Fill the buffer
        buffer = []
        for _ in range(self.buffer_size):
            try:
                buffer.append(next(it))
            except StopIteration:
                break

        # Yield a uniformly random buffered element, then put the next streamed one in its slot
        for item in it:
            idx = random.randint(0, len(buffer) - 1)
            yield buffer[idx]
            buffer[idx] = item

        # Drain the remaining buffer in random order
        random.shuffle(buffer)
        yield from buffer


# Demo
shuffled = ShuffleBufferDataset(SequentialDataset(20), buffer_size=5)
print("Shuffled order:", [idx.item() for idx, _ in shuffled])

In [ ]:
# === Tests ===

N = 500

# Test 1: All samples present, no duplicates
shuffled_ds = ShuffleBufferDataset(SequentialDataset(N), buffer_size=50)
indices = sorted([idx.item() for idx, _ in shuffled_ds])
assert indices == list(range(N)), "Not all samples yielded or duplicates detected"
print(f"PASSED: All {N} samples present, no duplicates")

# Test 2: Order is shuffled (not sequential)
shuffled_ds2 = ShuffleBufferDataset(SequentialDataset(N), buffer_size=50)
order = [idx.item() for idx, _ in shuffled_ds2]
assert order != list(range(N)), "Output is still in sequential order — not shuffled"
print("PASSED: Output order is shuffled")

# Test 3: Works with DataLoader
shuffled_ds3 = ShuffleBufferDataset(SequentialDataset(N), buffer_size=100)
loader = DataLoader(shuffled_ds3, batch_size=32)
total = sum(bi.shape[0] for bi, _ in loader)
assert total == N, f"DataLoader yielded {total} samples, expected {N}"
print("PASSED: Works with DataLoader batching")

# Test 4: Larger buffer → more shuffling
# Measure how far each element moved from its original position
def avg_displacement(ds):
    order = [idx.item() for idx, _ in ds]
    return sum(abs(i - order[i]) for i in range(len(order))) / len(order)

disp_small = avg_displacement(ShuffleBufferDataset(SequentialDataset(N), buffer_size=10))
disp_large = avg_displacement(ShuffleBufferDataset(SequentialDataset(N), buffer_size=200))
print(f"Avg displacement: buffer=10 → {disp_small:.1f}, buffer=200 → {disp_large:.1f}")
assert disp_large > disp_small, "Larger buffer should give more shuffling"
print("PASSED: Larger buffer produces more shuffling")

# Test 5: The first sample is a uniform draw from the first buffer_size elements
# (Tests 2 and 4 alone would pass a buffer that always picks slot 0 and only shuffles the final drain)
from collections import Counter
B, trials = 10, 2000
firsts = Counter(next(iter(ShuffleBufferDataset(SequentialDataset(50), buffer_size=B)))[0].item()
                 for _ in range(trials))
assert set(firsts) == set(range(B)), \
    f"The first sample should be able to be any of the first {B} elements, got {sorted(firsts)}"
assert trials / (2 * B) < min(firsts.values()) and max(firsts.values()) < 2 * trials / B, \
    f"The first sample is far from uniform over the buffer: {dict(sorted(firsts.items()))}"
print("PASSED: The first sample is drawn uniformly from the buffer")

# Test 6: O(buffer_size) memory - a sample cannot come out more than buffer_size positions
# before its position in the stream, because it has not been read yet
B = 25
order = [idx.item() for idx, _ in ShuffleBufferDataset(SequentialDataset(N), buffer_size=B)]
early = max(i - pos for pos, i in enumerate(order))
assert early <= B, (f"Element came out {early} positions before its stream position; a buffer of "
                    f"{B} cannot have read it yet - is the whole stream being loaded?")
print(f"PASSED: No sample jumps more than buffer_size={B} positions ahead")

# Test 7: A buffer larger than the dataset is just a full shuffle
small = [idx.item() for idx, _ in ShuffleBufferDataset(SequentialDataset(5), buffer_size=100)]
assert sorted(small) == list(range(5)), f"Expected each of 0..4 once, got {small}"
print("PASSED: buffer_size > dataset size works")

print("\nAll tests passed!")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("buffered-loading", ShuffleBufferDataset=ShuffleBufferDataset)

### Notes for the interview

- **It is only approximately shuffled.** Element `i` of the stream cannot come out before roughly position `i - buffer_size`, so samples from the end of the stream never appear early in an epoch, and with `buffer_size=B` neighbours in the stream tend to stay close. A full shuffle needs `buffer_size >= N`.
- **Shuffle at two levels in practice.** Large datasets are stored as many shards (files); shuffling the *order of the shards* each epoch and then running a shuffle buffer over the concatenated stream gives much better mixing for the same memory — this is what HuggingFace `IterableDataset.shuffle` and WebDataset do.
- **Multiple workers.** Each DataLoader worker runs its own copy of `__iter__` (and its own buffer), so the base dataset must shard by worker, and memory is `num_workers * buffer_size`. DataLoader seeds Python's `random` differently in every worker, so the buffers do not shuffle identically.
- **Reproducibility.** The global `random` module makes runs irreproducible; production code takes a seed and uses `random.Random(seed + epoch)` (plus the worker id), like `DistributedSampler.set_epoch`.